In [1]:
!pip install -U google-generativeai

In [10]:
import os
import sqlite3
import json
import time
from PIL import Image
from tqdm.auto import tqdm
from google.colab import userdata

# 1. 구글 드라이브 마운트
from google.colab import drive
drive.mount('/content/drive')

# 2. 최신버전 Gemini SDK 임포트
from google import genai
from google.genai import types

# API 키 세팅 및 최신 클라이언트 생성
GEMINI_API_KEY = userdata.get('GEMINI_API_KEY')
client = genai.Client(api_key=GEMINI_API_KEY)

DB_PATH = "/content/drive/MyDrive/CV_Project/seongdong_db/metadata_seongdong.db"
OUTPUT_FILE = "/content/drive/MyDrive/CV_Project/data/raw/tiles/metadata_gemini_finetune.jsonl"
BATCH_SIZE = 10

def process_image_batch(batch_rows):
    contents = []

    instruction = f"""
    당신은 프롭테크 입지 분석 AI입니다. 다음 제공되는 {len(batch_rows)}장의 위성 사진들을 각각 1문장으로 묘사하세요.
    수치를 쓰지 말고 건물의 밀집도, 녹지, 도로 형태 등 시각적 분위기를 묘사하세요.
    결과는 반드시 아래의 JSON Array 포맷으로만 출력하세요:
    [
      {{"file_name": "이미지파일명1", "text": "묘사 내용"}},
      {{"file_name": "이미지파일명2", "text": "묘사 내용"}}
    ]
    """
    contents.append(instruction)

    valid_rows = []

    BASE_DIR = "/content/drive/MyDrive/CV_Project"

    for row in batch_rows:
        tile_id, district, db_image_path = row

        clean_path = db_image_path.replace("\\", "/")
        image_path = os.path.join(BASE_DIR, clean_path)

        if not os.path.exists(image_path):
            print(f"⚠️ 파일 없음 스킵: {image_path}") # 혹시 파일이 없으면 이유를 출력
            continue

        try:
            img = Image.open(image_path).convert('RGB')
            contents.append(f"파일명: {os.path.basename(image_path)}, 위치: {district}")
            contents.append(img)
            valid_rows.append(row)
        except Exception as e:
            print(f"이미지 로드 실패 ({image_path}): {e}")

    if not valid_rows:
        return []

    MAX_RETRIES = 5
    for attempt in range(MAX_RETRIES):
        try:
            response = client.models.generate_content(
                model='gemini-3.1-flash-lite',
                contents=contents,
                config=types.GenerateContentConfig(
                    response_mime_type="application/json",
                    temperature=0.2,
                )
            )
            results = json.loads(response.text)
            return results

        except Exception as e:
            error_str = str(e)
            if '503' in error_str or 'UNAVAILABLE' in error_str:
                wait = (attempt + 1) * 10  # 10초, 20초, 30초...
                print(f"⏳ 503 에러, {wait}초 후 재시도... ({attempt+1}/{MAX_RETRIES})")
                time.sleep(wait)
            else:
                print(f"\n🚨 API 호출 또는 파싱 에러: {e}")
                return []

    print("❌ 최대 재시도 횟수 초과, 배치 스킵")
    return []

def main():
    if not GEMINI_API_KEY:
        print("❌ GEMINI_API_KEY가 설정되지 않았습니다.")
        return

    print("1. SQLite DB에서 이미지 목록 로드 중...")
    conn = sqlite3.connect(DB_PATH)
    cursor = conn.cursor()
    cursor.execute("SELECT tile_id, district, image_path FROM tile_info")

    rows = cursor.fetchall()
    conn.close()

    print(f"2. 총 {len(rows)}장 타일: 한 번에 {BATCH_SIZE}장씩 최신 Gemini API로 다중 분석 시작...")

    all_results = []

    for i in tqdm(range(0, len(rows), BATCH_SIZE), desc="배치 처리 중"):
        batch_rows = rows[i:i+BATCH_SIZE]

        batch_results = process_image_batch(batch_rows)
        if batch_results:
            all_results.extend(batch_results)

        time.sleep(5)

    print(f"\n3. 추출 완료! 총 {len(all_results)}개의 캡션 결과를 저장합니다...")
    os.makedirs(os.path.dirname(OUTPUT_FILE), exist_ok=True)
    with open(OUTPUT_FILE, "w", encoding="utf-8") as f:
        for res in all_results:
            f.write(json.dumps(res, ensure_ascii=False) + "\n")

    print(f"🎉 성공적으로 최신 API를 통해 데이터셋이 생성되었습니다: {OUTPUT_FILE}")

if __name__ == "__main__":
    main()

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
1. SQLite DB에서 이미지 목록 로드 중...
2. 총 1824장 타일: 한 번에 10장씩 최신 Gemini API로 다중 분석 시작...


배치 처리 중:   0%|          | 0/183 [00:00<?, ?it/s]

⏳ 503 에러, 10초 후 재시도... (1/5)
⏳ 503 에러, 10초 후 재시도... (1/5)
⏳ 503 에러, 10초 후 재시도... (1/5)
⏳ 503 에러, 10초 후 재시도... (1/5)

🚨 API 호출 또는 파싱 에러: Expecting ':' delimiter: line 9 column 31 (char 792)

🚨 API 호출 또는 파싱 에러: Expecting ':' delimiter: line 7 column 31 (char 613)

3. 추출 완료! 총 1804개의 캡션 결과를 저장합니다...
🎉 성공적으로 최신 API를 통해 데이터셋이 생성되었습니다: /content/drive/MyDrive/CV_Project/data/raw/tiles/metadata_gemini_finetune.jsonl


In [8]:
for model in client.models.list():
    print(model.name)

models/gemini-2.5-flash
models/gemini-2.5-pro
models/gemini-2.0-flash
models/gemini-2.0-flash-001
models/gemini-2.0-flash-lite-001
models/gemini-2.0-flash-lite
models/gemini-2.5-flash-preview-tts
models/gemini-2.5-pro-preview-tts
models/gemma-4-26b-a4b-it
models/gemma-4-31b-it
models/gemini-flash-latest
models/gemini-flash-lite-latest
models/gemini-pro-latest
models/gemini-2.5-flash-lite
models/gemini-2.5-flash-image
models/gemini-3-pro-preview
models/gemini-3-flash-preview
models/gemini-3.1-pro-preview
models/gemini-3.1-pro-preview-customtools
models/gemini-3.1-flash-lite-preview
models/gemini-3.1-flash-lite
models/gemini-3-pro-image-preview
models/gemini-3-pro-image
models/nano-banana-pro-preview
models/gemini-3.1-flash-image-preview
models/gemini-3.1-flash-image
models/gemini-3.5-flash
models/lyria-3-clip-preview
models/lyria-3-pro-preview
models/gemini-3.1-flash-tts-preview
models/gemini-robotics-er-1.5-preview
models/gemini-robotics-er-1.6-preview
models/gemini-2.5-computer-use-pr

In [2]:
# 기존 torchao 완전 제거 후 재설치
!pip uninstall torchao -y
!pip install torchao==0.16.0 --force-reinstall --no-deps

Found existing installation: torchao 0.10.0
Uninstalling torchao-0.10.0:
  Successfully uninstalled torchao-0.10.0
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 54.0 MB/s eta 0:00:00


In [2]:
# ============================================================
# 0. 설치
# ============================================================
!pip install -q transformers datasets peft accelerate torchao --upgrade

# ============================================================
# 1. 임포트 & 드라이브 마운트
# ============================================================
import os, json, torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from transformers import CLIPProcessor, CLIPModel
from peft import get_peft_model, LoraConfig
from torch.cuda.amp import autocast, GradScaler
from google.colab import drive

drive.mount('/content/drive')

# ============================================================
# 2. 경로 설정
# ============================================================
MODEL_DIR  = "/content/drive/MyDrive/CV_Project/my_large_satellite_clip"
JSONL_PATH = "/content/drive/MyDrive/CV_Project/data/raw/tiles/metadata_gemini_finetune.jsonl"
IMAGE_DIR  = "/content/drive/MyDrive/CV_Project/data/raw/tiles"
OUTPUT_DIR = "/content/drive/MyDrive/CV_Project/my_large_satellite_clip_v2"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ============================================================
# 3. 데이터셋
# ============================================================
class SatelliteDataset(Dataset):
    def __init__(self, jsonl_path, image_dir, processor):
        self.processor = processor

        # ✅ os.walk를 __init__에서 딱 한 번만 실행해 파일명→경로 캐싱
        print("이미지 경로 인덱싱 중...")
        self.image_index = {}
        for root, _, files in os.walk(image_dir):
            for fname in files:
                if fname.endswith(".jpg") or fname.endswith(".png"):
                    self.image_index[fname] = os.path.join(root, fname)
        print(f"인덱싱 완료: {len(self.image_index)}개 이미지")

        # jsonl 로드 (인덱스에 있는 것만)
        self.samples = []
        with open(jsonl_path, "r", encoding="utf-8") as f:
            for line in f:
                item = json.loads(line)
                fname = item["file_name"]
                if fname in self.image_index:
                    self.samples.append({
                        "image_path": self.image_index[fname],
                        "text": item["text"]
                    })
                else:
                    print(f"⚠️ 이미지 없음 스킵: {fname}")
        print(f"학습 샘플 수: {len(self.samples)}")

    def __len__(self):
        return len(self.samples)

    def __getitem__(self, idx):
        # ✅ __getitem__에서는 딕셔너리 조회만 (O(1))
        s = self.samples[idx]
        image = Image.open(s["image_path"]).convert("RGB")
        inputs = self.processor(
            text=s["text"],
            images=image,
            return_tensors="pt",
            padding="max_length",
            truncation=True,
            max_length=77
        )
        return {k: v.squeeze(0) for k, v in inputs.items()}

# ============================================================
# 4. 모델 & LoRA 설정
# ============================================================
print("모델 로딩 중...")
from transformers import CLIPProcessor, CLIPModel
import os

MODEL_DIR = os.path.abspath("/content/drive/MyDrive/CV_Project/my_large_satellite_clip")
processor = CLIPProcessor.from_pretrained(MODEL_DIR, local_files_only=True)
model     = CLIPModel.from_pretrained(MODEL_DIR, local_files_only=True, torch_dtype=torch.float32)

lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj", "v_proj"],
    lora_dropout=0.05,
    bias="none",
)
model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

model.gradient_checkpointing_enable()
model = model.cuda()

# ============================================================
# 5. DataLoader
# ============================================================
dataset    = SatelliteDataset(JSONL_PATH, IMAGE_DIR, processor)
dataloader = DataLoader(
    dataset,
    batch_size=8,
    shuffle=True,
    num_workers=2,
    pin_memory=True
)

# ============================================================
# 6. 학습 설정
# ============================================================
optimizer = torch.optim.AdamW(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-4,
    weight_decay=0.01
)
scaler = GradScaler()

NUM_EPOCHS      = 5
LOG_EVERY_STEPS = 20

# ============================================================
# 7. Contrastive Loss
# ============================================================
def contrastive_loss(image_embeds, text_embeds, logit_scale):
    image_embeds = image_embeds / image_embeds.norm(dim=-1, keepdim=True)
    text_embeds  = text_embeds  / text_embeds.norm(dim=-1, keepdim=True)

    logits_per_image = logit_scale * image_embeds @ text_embeds.T
    logits_per_text  = logits_per_image.T

    labels = torch.arange(len(image_embeds), device=image_embeds.device)
    loss_i = torch.nn.functional.cross_entropy(logits_per_image, labels)
    loss_t = torch.nn.functional.cross_entropy(logits_per_text,  labels)
    return (loss_i + loss_t) / 2

# ============================================================
# 8. 학습 루프
# ============================================================
print("학습 시작!")
model.train()

for epoch in range(NUM_EPOCHS):
    total_loss = 0.0

    for step, batch in enumerate(dataloader):
        batch = {k: v.cuda() for k, v in batch.items()}
        optimizer.zero_grad()

        with autocast():
            outputs = model(**batch)

            # ✅ LoRA base_model에서 logit_scale 명확하게 접근
            logit_scale = model.base_model.model.logit_scale.exp()

            loss = contrastive_loss(
                outputs.image_embeds,
                outputs.text_embeds,
                logit_scale
            )

        scaler.scale(loss).backward()
        scaler.unscale_(optimizer)
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        scaler.step(optimizer)
        scaler.update()

        total_loss += loss.item()
        if (step + 1) % LOG_EVERY_STEPS == 0:
            avg = total_loss / (step + 1)
            print(f"[Epoch {epoch+1}/{NUM_EPOCHS}] Step {step+1} | Loss: {avg:.4f}")

    print(f"✅ Epoch {epoch+1} 완료 | 평균 Loss: {total_loss/len(dataloader):.4f}")

    ckpt_path = os.path.join(OUTPUT_DIR, f"epoch_{epoch+1}")
    model.save_pretrained(ckpt_path)
    processor.save_pretrained(ckpt_path)
    print(f"💾 저장 완료: {ckpt_path}")

print("🎉 파인튜닝 완료!")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.0/11.0 MB 110.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 555.1/555.1 kB 35.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.2/3.2 MB 115.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 48.9/48.9 MB 20.5 MB/s eta 0:00:00
Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
모델 로딩 중...


Loading weights:   0%|          | 0/590 [00:00<?, ?it/s]

trainable params: 1,081,344 || all params: 428,697,857 || trainable%: 0.2522
이미지 경로 인덱싱 중...
인덱싱 완료: 1824개 이미지
학습 샘플 수: 1804
학습 시작!


/tmp/ipykernel_1540/3853198389.py:120: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
/tmp/ipykernel_1540/3853198389.py:153: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():


[Epoch 1/5] Step 20 | Loss: 1.9279
[Epoch 1/5] Step 40 | Loss: 1.8957
[Epoch 1/5] Step 60 | Loss: 1.7391
[Epoch 1/5] Step 80 | Loss: 1.6285
[Epoch 1/5] Step 100 | Loss: 1.5279
[Epoch 1/5] Step 120 | Loss: 1.4436
[Epoch 1/5] Step 140 | Loss: 1.3674
[Epoch 1/5] Step 160 | Loss: 1.3114
[Epoch 1/5] Step 180 | Loss: 1.2459
[Epoch 1/5] Step 200 | Loss: 1.2063
[Epoch 1/5] Step 220 | Loss: 1.1688
✅ Epoch 1 완료 | 평균 Loss: 1.1571
💾 저장 완료: /content/drive/MyDrive/CV_Project/my_large_satellite_clip_v2/epoch_1
[Epoch 2/5] Step 20 | Loss: 0.6793
[Epoch 2/5] Step 40 | Loss: 0.6708
[Epoch 2/5] Step 60 | Loss: 0.6823
[Epoch 2/5] Step 80 | Loss: 0.7051
[Epoch 2/5] Step 100 | Loss: 0.6792
[Epoch 2/5] Step 120 | Loss: 0.6625
[Epoch 2/5] Step 140 | Loss: 0.6528
[Epoch 2/5] Step 160 | Loss: 0.6452
[Epoch 2/5] Step 180 | Loss: 0.6576
[Epoch 2/5] Step 200 | Loss: 0.6530
[Epoch 2/5] Step 220 | Loss: 0.6425
✅ Epoch 2 완료 | 평균 Loss: 0.6481
💾 저장 완료: /content/drive/MyDrive/CV_Project/my_large_satellite_clip_v2/epoch_